# Supervised Fine-Tuning From Scratch, Then With TRL

Build the label mask, the masked loss and document-masked packing by hand on SmolLM2-135M, and prove each equals TRL. Then fully fine-tune SmolLM2-360M on a free T4 and measure what each choice changes.

LLM Training Lab, module 11. Regime: SFT. Steps 1 to 7 run on a CPU with SmolLM2-135M. Step 8 fine-tunes SmolLM2-360M on a T4. The cells below are the lab page's code, in order.

In [ ]:
# Runtime check. Run this cell first.
import os, platform, shutil, subprocess, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # before torch loads: SFTTrainer refuses a model split over 2 GPUs
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    print("PROBLEM Python 3.11 or later is needed, 3.12 recommended. Start a current Colab or Kaggle session.")
free = shutil.disk_usage(".").free / 1e9
print("Free disk", round(free, 1), "GB")
if free < 5:
    print("PROBLEM steps 1 to 8 download about 1.3 GB of models and stream the data. Free some disk.")
try:   # a CPU-only runtime has no nvidia-smi at all
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("GPU", gpu or "none")
if not gpu:
    print("Steps 1 to 7 run on this CPU. Step 8 needs a GPU: Colab, Runtime > Change runtime type > T4 GPU. Kaggle, Settings > Accelerator > GPU T4 x2.")
elif "P100" in gpu:
    print("PROBLEM a P100 has no kernels in current PyTorch. Pick a T4 instead.")
from importlib.metadata import version, PackageNotFoundError
for pkg, want in [("transformers", "5.22.1"), ("trl", "1.17.1"), ("accelerate", "1.15.0"), ("datasets", "5.1.0")]:
    try:
        have = version(pkg)
    except PackageNotFoundError:
        have = "not installed"
    print(pkg, have, "" if have == want else "(the next cell installs " + want + ")")
print("Runtime OK" if sys.version_info >= (3, 11) else "Fix the PROBLEM lines, then run this cell again.")

In [ ]:
# Pinned Hugging Face versions of our recorded runs. torch and numpy stay as the runtime ships them.
# If transformers was already imported in this session, restart the session after this cell, then run from step 1.
%pip install -q "transformers==5.22.1" "trl==1.17.1" "accelerate==1.15.0" "datasets==5.1.0" "tokenizers==0.23.3" "huggingface_hub==2.4.0"

The next cell writes `sft_common.py`. It holds the same selection and label mask as steps 2 and 3, and the two evaluation helpers that step 8 imports.

In [ ]:
%%writefile sft_common.py
"""Shared code for the lltSFT lab: data selection, the training chat template, the label mask and the evals.

Used by measure.py (CPU, T1) and worker_sft.py (Kaggle T4, T2), so every run selects the same conversations
and scores them the same way. The lab page shows the same functions in its build steps.

Data: HuggingFaceTB/smoltalk2, config SFT, three no_think splits rebuilt from SmolTalk's Smol-Magpie-Ultra,
Smol-rewrite and Smol-summarize (Apache-2.0 per the SmolTalk card). Streamed in file order, so the selection is
deterministic. A conversation is kept when it renders to at most MAX_TOKENS tokens under the SmolLM2 template.
The first N_EVAL kept rows of each split are the held-out eval set, the next N_TRAIN the training set.
"""
import hashlib
import itertools
import json
import math

import torch

BASE_360 = "HuggingFaceTB/SmolLM2-360M"
BASE_135 = "HuggingFaceTB/SmolLM2-135M"
INSTRUCT_360 = "HuggingFaceTB/SmolLM2-360M-Instruct"
INSTRUCT_135 = "HuggingFaceTB/SmolLM2-135M-Instruct"
DATASET = "HuggingFaceTB/smoltalk2"
SPLITS = [
    "smoltalk_smollm3_smol_magpie_ultra_no_think",
    "smoltalk_smollm3_smol_rewrite_no_think",
    "smoltalk_smollm3_smol_summarize_no_think",
]
MAX_TOKENS = 1024
N_EVAL = 100
N_TRAIN = 1200
N_GEN = 20          # first N_GEN eval rows of each split also give a generation prompt


def to_messages(ex):
    """SmolTalk2 stores the system prompt in chat_template_kwargs.custom_instructions, not in messages."""
    msgs = [{"role": m["role"], "content": m["content"]} for m in ex["messages"]]
    system = (ex.get("chat_template_kwargs") or {}).get("custom_instructions") or ""
    if system and msgs[0]["role"] != "system":
        msgs = [{"role": "system", "content": system}] + msgs
    return msgs


def training_template(instruct_template):
    """The instruct template with {% generation %} markers around each assistant reply and its <|im_end|>.

    Renders the same text as the instruct template. TRL's assistant_only_loss needs the markers.
    """
    t = instruct_template.replace(
        "' + message['content'] + '<|im_end|>' + '",
        "'}}{% if message['role'] == 'assistant' %}{% generation %}{{ message['content'] + '<|im_end|>' }}"
        "{% endgeneration %}{% else %}{{ message['content'] + '<|im_end|>' }}{% endif %}{{'")
    assert t != instruct_template, "the instruct template changed; update training_template"
    return t


def setup_tokenizer(tok, instruct_tok):
    """Base SmolLM2 ships no chat template: borrow the instruct one, make <|im_end|> the EOS, keep pad apart from EOS."""
    tok.chat_template = training_template(instruct_tok.chat_template)
    tok.eos_token = "<|im_end|>"
    tok.pad_token = "<|endoftext|>"
    return tok


def header_ids(tok):
    """The assistant header, <|im_start|> assistant newline: the last 4 ids of the generation prompt (ASCII-only source)."""
    gen = tok.apply_chat_template([{"role": "user", "content": "x"}], add_generation_prompt=True)
    return list(gen["input_ids"])[-4:]


def build_labels(ids, header, im_end):
    """-100 everywhere except each assistant reply and its <|im_end|>."""
    labels = [-100] * len(ids)
    t = 0
    while t < len(ids):
        if ids[t:t + len(header)] == header:
            t += len(header)
            while t < len(ids):
                labels[t] = ids[t]
                t += 1
                if ids[t - 1] == im_end:
                    break
        else:
            t += 1
    return labels


def select_conversations(tok, n_eval=N_EVAL, n_train=N_TRAIN, max_tokens=MAX_TOKENS, splits=SPLITS, log=print):
    """Returns (train, eval, stats). Each item: {"split", "row", "messages", "n_tokens"}."""
    from datasets import load_dataset
    train, held, stats = [], [], {}
    for split in splits:
        ds = load_dataset(DATASET, "SFT", split=split, streaming=True)
        kept, seen = [], 0
        for row, ex in enumerate(ds):
            seen += 1
            msgs = to_messages(ex)
            n = len(tok.apply_chat_template(msgs)["input_ids"])
            if n <= max_tokens:
                kept.append({"split": split, "row": row, "messages": msgs, "n_tokens": n})
                if len(kept) == n_eval + n_train:
                    break
        held += kept[:n_eval]
        train += kept[n_eval:]
        stats[split] = {"rowsRead": seen, "kept": len(kept), "keptShare": round(len(kept) / seen, 4)}
        log(split, stats[split])
    return train, held, stats


def data_hash(items):
    h = hashlib.sha256()
    for it in items:
        h.update(json.dumps(it["messages"], ensure_ascii=False, sort_keys=True).encode("utf-8"))
    return h.hexdigest()


@torch.no_grad()
def eval_loss(model, tok, convs, device, batch_size=8, dtype=None):
    """Token-weighted masked NLL over the assistant tokens of each held-out conversation, one conversation per row."""
    header, im_end = header_ids(tok), tok.convert_tokens_to_ids("<|im_end|>")
    was_training = model.training
    model.eval()
    nll, count = 0.0, 0
    for i in range(0, len(convs), batch_size):
        chunk = convs[i:i + batch_size]
        ids = [list(tok.apply_chat_template(c["messages"])["input_ids"]) for c in chunk]
        T = max(len(x) for x in ids)
        x = torch.full((len(ids), T), tok.pad_token_id, dtype=torch.long)
        y = torch.full((len(ids), T), -100, dtype=torch.long)
        am = torch.zeros((len(ids), T), dtype=torch.long)
        for r, row in enumerate(ids):
            x[r, :len(row)] = torch.tensor(row)
            y[r, :len(row)] = torch.tensor(build_labels(row, header, im_end))
            am[r, :len(row)] = 1
        x, y, am = x.to(device), y.to(device), am.to(device)
        with torch.autocast(device.type if hasattr(device, "type") else str(device), dtype=dtype, enabled=dtype is not None):
            logits = model(input_ids=x, attention_mask=am, use_cache=False).logits
        tgt = y[:, 1:]
        m = tgt != -100
        nll += torch.nn.functional.cross_entropy(logits[:, :-1][m].float(), tgt[m], reduction="sum").item()
        count += int(m.sum())
    if was_training:
        model.train()
    return {"loss": nll / count, "tokens": count}


@torch.no_grad()
def eval_generate(model, tok, convs, device, max_new_tokens=512, batch_size=20, dtype=None, template=True):
    """Greedy answers to the first user turn (with its system prompt). Stop rate = share that end with <|im_end|>."""
    im_end = tok.convert_tokens_to_ids("<|im_end|>")
    was_training = model.training
    model.eval()
    prompts = []
    for c in convs:
        msgs = c["messages"]
        first = [m for m in msgs[:2] if m["role"] == "system"] + [next(m for m in msgs if m["role"] == "user")]
        if template:
            prompts.append(list(tok.apply_chat_template(first, add_generation_prompt=True)["input_ids"]))
        else:
            prompts.append(list(tok("\n\n".join(m["content"] for m in first) + "\n\n")["input_ids"]))
    out = []
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        T = max(len(p) for p in chunk)
        x = torch.full((len(chunk), T), tok.pad_token_id, dtype=torch.long)
        am = torch.zeros((len(chunk), T), dtype=torch.long)
        for r, p in enumerate(chunk):          # left padding for generation
            x[r, T - len(p):] = torch.tensor(p)
            am[r, T - len(p):] = 1
        x, am = x.to(device), am.to(device)
        with torch.autocast(device.type if hasattr(device, "type") else str(device), dtype=dtype, enabled=dtype is not None):
            g = model.generate(input_ids=x, attention_mask=am, max_new_tokens=max_new_tokens, do_sample=False,
                               eos_token_id=im_end, pad_token_id=tok.pad_token_id)
        for r in range(len(chunk)):
            new = g[r, T:].tolist()
            stopped = im_end in new
            n = new.index(im_end) + 1 if stopped else len([t for t in new if t != tok.pad_token_id])
            out.append({"stopped": stopped, "newTokens": n, "text": tok.decode(new[:n], skip_special_tokens=False)})
    if was_training:
        model.train()
    stopped = [o for o in out if o["stopped"]]
    return {
        "n": len(out), "stopRate": round(len(stopped) / len(out), 4),
        "meanNewTokens": round(sum(o["newTokens"] for o in out) / len(out), 1),
        "meanNewTokensStopped": round(sum(o["newTokens"] for o in stopped) / len(stopped), 1) if stopped else None,
        "samples": out,
    }


def generation_set(held, n_per_split=N_GEN, splits=SPLITS):
    return [c for s in splits for c in [h for h in held if h["split"] == s][:n_per_split]]


def cross_doc_collator(collate, restart_positions, explicit_mask=True):
    # Same packed rows and labels as TRL's collator, but every token may attend to every earlier token in the row.
    # explicit_mask=True passes a full causal mask, so SDPA gets an explicit mask, as it does with the document mask.
    def wrapped(examples):
        b = collate(examples)
        n = b["input_ids"].shape[1]
        if not restart_positions:
            b["position_ids"] = torch.arange(n)[None]
        if explicit_mask:
            b["attention_mask"] = torch.tril(torch.ones(n, n, dtype=torch.bool))[None, None]
        for k in ("cu_seq_lens_q", "cu_seq_lens_k", "max_length_q", "max_length_k", "seq_idx"):
            b.pop(k, None)
        return b
    return wrapped

## Step 1. Give the base model a chat template

What you should see. The base tokenizer prints no template, EOS `<|endoftext|>` and pad None. The instruct tokenizer prints EOS `<|im_end|>` with id 2, and its pad is the same token, id 2. The rendered chat has a system turn you did not write, **You are a helpful AI assistant named SmolLM, trained by Hugging Face**. The chat is 39 tokens, and the first 21 are that system turn.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle has 2 GPUs, and SFTTrainer refuses to split across them
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_num_threads(4)
BASE = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(BASE)
inst = AutoTokenizer.from_pretrained(BASE + "-Instruct")
print("base chat template:", tok.chat_template is not None, "| base EOS", tok.eos_token, "| base pad", tok.pad_token)
print("instruct EOS", inst.eos_token, inst.eos_token_id, "| instruct pad", inst.pad_token, inst.pad_token_id)

tok.chat_template = inst.chat_template   # borrow it: both tokenizers share all 49,152 tokens
tok.eos_token = "<|im_end|>"             # the template ends every turn with this token
tok.pad_token = "<|endoftext|>"          # keep padding apart from EOS

chat = [{"role": "user", "content": "Name a fruit."}, {"role": "assistant", "content": "A mango."}]
print(tok.apply_chat_template(chat, tokenize=False))
ids = list(tok.apply_chat_template(chat)["input_ids"])
print(len(ids), "tokens, the first", ids.index(1, 1), "of them the system prompt you did not write")

## Step 2. Load the conversations and keep their system prompts

What you should see. The cell prints the rows read per split, 9,010, 1,300 and 1,547, then 3,600 training and 300 held-out conversations. 2,400 training conversations carry a system prompt. Without `to_messages`, every one of them would be lost.

In [ ]:
from datasets import load_dataset

def to_messages(ex):
    # SmolTalk2 keeps the system prompt in chat_template_kwargs, not in messages
    msgs = [{"role": m["role"], "content": m["content"]} for m in ex["messages"]]
    system = (ex.get("chat_template_kwargs") or {}).get("custom_instructions") or ""
    if system:
        msgs = [{"role": "system", "content": system}] + msgs
    return msgs

SPLITS = ["smoltalk_smollm3_smol_magpie_ultra_no_think", "smoltalk_smollm3_smol_rewrite_no_think",
          "smoltalk_smollm3_smol_summarize_no_think"]

def select(split, n_eval=100, n_train=1200, max_tokens=1024):
    kept = []
    for row, ex in enumerate(load_dataset("HuggingFaceTB/smoltalk2", "SFT", split=split, streaming=True)):
        msgs = to_messages(ex)
        n = len(tok.apply_chat_template(msgs)["input_ids"])
        if n <= max_tokens:
            kept.append({"split": split, "row": row, "messages": msgs, "n_tokens": n})
        if len(kept) == n_eval + n_train:
            print(split, "read", row + 1, "rows, kept", len(kept))
            return kept[:n_eval], kept[n_eval:]

held, train = [], []
for split in SPLITS:
    e, t = select(split)
    held += e
    train += t
print(len(train), "training and", len(held), "held-out conversations")
print("system prompts lost without to_messages:", sum(m["messages"][0]["role"] == "system" for m in train))

## Step 3. Build the label mask

What you should see. The first line prints the header ids [1, 520, 9531, 198]. The printout below it shows the system and user turns as plain text and each reply token in brackets, ending with `[<|im_end|>]`. The newline after it stays unbracketed. The shares with loss are 0.799 for Magpie, 0.427 for rewrite and 0.196 for summarize. Over all 1,797,589 training tokens the share is 0.583 (lltSFT-t1-data).

In [ ]:
gen = tok.apply_chat_template([{"role": "user", "content": "x"}], add_generation_prompt=True)
HEADER = list(gen["input_ids"])[-4:]          # the generation prompt is the assistant header
print("assistant header", HEADER, tok.convert_ids_to_tokens(HEADER))
IM_END = tok.convert_tokens_to_ids("<|im_end|>")

def build_labels(ids):
    labels = [-100] * len(ids)                 # -100 means no loss at this position
    t = 0
    while t < len(ids):
        if ids[t:t + 4] == HEADER:              # an assistant reply starts after its header
            t += 4
            while t < len(ids):
                labels[t] = ids[t]              # train on the reply ...
                t += 1
                if ids[t - 1] == IM_END:        # ... up to and including its end-of-turn token
                    break
        else:
            t += 1
    return labels

ex = train[1200]["messages"]                    # the first rewrite example, which has a system prompt
ids = list(tok.apply_chat_template(ex)["input_ids"])
labels = build_labels(ids)
shown = "".join(("[" + tok.decode([i]) + "]") if l != -100 else tok.decode([i]) for i, l in zip(ids, labels))
print(shown[:260], "...", shown[-90:])
for k, split in enumerate(SPLITS):
    n_tok = n_loss = 0
    for c in train[1200 * k:1200 * (k + 1)]:
        ids = list(tok.apply_chat_template(c["messages"])["input_ids"])
        n_tok += len(ids)
        n_loss += sum(l != -100 for l in build_labels(ids))
    print(split, "| tokens", n_tok, "| share with loss", round(n_loss / n_tok, 3))

## Step 4. Compute the masked loss by hand

What you should see. The rewrite example has 124 assistant tokens. Your loss and the transformers loss both print 1.78691. The loss over all 265 predicted tokens is 2.81612, so on this example the prompt tokens score higher on average than the reply.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).eval()

def masked_nll(model, ids, labels):
    x, y = torch.tensor([ids]), torch.tensor(labels)
    with torch.no_grad():
        logits = model(input_ids=x, use_cache=False).logits[0]
    pred, target = logits[:-1], y[1:]           # the logits at position t predict token t + 1
    keep = target != -100
    return torch.nn.functional.cross_entropy(pred[keep], target[keep]).item(), int(keep.sum())

ids = list(tok.apply_chat_template(train[1200]["messages"])["input_ids"])
labels = build_labels(ids)
own, n = masked_nll(model, ids, labels)
with torch.no_grad():
    lib = model(input_ids=torch.tensor([ids]), labels=torch.tensor([labels]), use_cache=False).loss.item()
every, n_all = masked_nll(model, ids, ids)       # the same example, loss on every token
print("assistant tokens", n, "| own", round(own, 5), "| transformers", round(lib, 5))
print("all", n_all, "tokens | loss", round(every, 5))

## Step 5. Check your mask and loss against TRL

What you should see. The new template renders the same ids. The labels are equal on a batch of shape [4, 890]. Your loss in float64 is 1.4967340850 and TRL's compute_loss gives 1.4967341423, a difference of 5.7e-08 on our laptop and 7.1e-08 on a Kaggle CPU.

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

# TRL finds assistant tokens through {% generation %} markers in the template. SmolLM2's has none, so add them.
tok.chat_template = inst.chat_template.replace(
    "' + message['content'] + '<|im_end|>' + '",
    "'}}{% if message['role'] == 'assistant' %}{% generation %}{{ message['content'] + '<|im_end|>' }}"
    "{% endgeneration %}{% else %}{{ message['content'] + '<|im_end|>' }}{% endif %}{{'")
print("renders the same ids:", list(tok.apply_chat_template(chat)["input_ids"]) == list(inst.apply_chat_template(chat)["input_ids"]))

args = SFTConfig(output_dir="sft-check", report_to="none", use_cpu=True, bf16=False,
                 assistant_only_loss=True, max_length=1024, per_device_train_batch_size=4)
eight = [{"messages": c["messages"]} for c in train[1200:1204] + train[:4]]
trainer = SFTTrainer(model=model, args=args, processing_class=tok, train_dataset=Dataset.from_list(eight))
batch = next(iter(trainer.get_train_dataloader()))

ours = torch.full_like(batch["input_ids"], -100)
for r in range(len(ours)):
    n = int(batch["attention_mask"][r].sum())
    ours[r, :n] = torch.tensor(build_labels(batch["input_ids"][r, :n].tolist()))
print("labels equal:", torch.equal(ours, batch["labels"]), "| batch", list(batch["input_ids"].shape))

with torch.no_grad():
    logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], use_cache=False).logits
    target = ours[:, 1:]
    keep = target != -100
    own = torch.nn.functional.cross_entropy(logits[:, :-1][keep].double(), target[keep]).item()
    lib = trainer.compute_loss(trainer.model, dict(batch)).item()
print("own", own, "| TRL", lib, "| difference", abs(own - lib))

## Step 6. Pack conversations without leaking

What you should see. Alone and packed with restarted positions give the same losses, 1.7869, 1.7672 and 1.9604. Naive packing leaves conversation 1 unchanged and gives 1.6344 and 1.8997 for the other two. With the cache on, the restarted positions no longer mask anything and the losses jump to 3.1706 and 4.2893. The last lines print `use_cache` False and the 5 by 5 mask of two documents of 3 and 2 tokens.

In [ ]:
docs = [list(tok.apply_chat_template(c["messages"])["input_ids"]) for c in train[1200:1203]]
lens = [len(d) for d in docs]
row = torch.tensor([sum(docs, [])])
reset = torch.cat([torch.arange(n) for n in lens])[None]   # positions restart at 0 in each conversation
straight = torch.arange(row.shape[1])[None]                 # naive packing, one long document

def losses(logits):
    out, start = [], 0
    for d in docs:
        y = torch.tensor(build_labels(d))
        keep = y[1:] != -100
        seg = logits[0, start:start + len(d) - 1]
        out.append(round(torch.nn.functional.cross_entropy(seg[keep], y[1:][keep]).item(), 4))
        start += len(d)
    return out

with torch.no_grad():
    print("alone          ", [round(masked_nll(model, d, build_labels(d))[0], 4) for d in docs])
    print("packed, reset  ", losses(model(input_ids=row, position_ids=reset, use_cache=False).logits))
    print("packed, naive  ", losses(model(input_ids=row, position_ids=straight, use_cache=False).logits))
    print("reset, cache on", losses(model(input_ids=row, position_ids=reset, use_cache=True).logits))
print("use_cache in the config after step 5:", model.config.use_cache)

doc_of = torch.tensor([0, 0, 0, 1, 1])                       # two documents of 3 and 2 tokens
causal = torch.tril(torch.ones(5, 5, dtype=torch.bool))
print((causal & (doc_of[:, None] == doc_of[None, :])).int())

## Step 7. Write the SFT loop by hand

What you should see. The cell prints 24 packed rows and a held-out loss of 1.8262 before training. The training loss prints every 8 steps, 1.7746 at step 0, 1.4583 at step 8 and 1.3747 at step 16. The held-out loss after the epoch is 1.4856. On our laptop CPU the cell took [[TIME:hand_loop_min]] min.

In [ ]:
torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.0)

def pack(convs, max_len=1024):
    rows, cur = [], []
    for c in convs:
        ids = list(tok.apply_chat_template(c["messages"])["input_ids"])
        if cur and sum(len(d) for d in cur) + len(ids) > max_len:
            rows.append(cur)
            cur = []
        cur.append(ids)
    return rows + [cur]

def heldout_loss(model, convs):
    total = count = 0
    for c in convs:
        ids = list(tok.apply_chat_template(c["messages"])["input_ids"])
        loss, n = masked_nll(model, ids, build_labels(ids))
        total += loss * n
        count += n
    return total / count

small = train[1200:1224] + train[2400:2424]                  # 48 short rewrite and summarize examples
check = held[100:124] + held[200:224]                         # 48 held-out ones from the same splits
rows = pack(small)
model.eval()
print(len(rows), "packed rows | held-out loss before", round(heldout_loss(model, check), 4))
model.train()
for step, docs in enumerate(rows):
    x = torch.tensor([sum(docs, [])])
    pos = torch.tensor([sum((list(range(len(d))) for d in docs), [])])
    y = torch.tensor(sum((build_labels(d) for d in docs), []))
    logits = model(input_ids=x, position_ids=pos, use_cache=False).logits[0]   # use_cache=False keeps the document mask
    keep = y[1:] != -100
    loss = torch.nn.functional.cross_entropy(logits[:-1][keep], y[1:][keep])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    opt.zero_grad()
    if step % 8 == 0:
        print("step", step, "train loss", round(loss.item(), 4))
model.eval()
print("held-out loss after", round(heldout_loss(model, check), 4))

## Step 8. Fine-tune SmolLM2-360M with TRL on a T4

What you should see. The base model scores 1.4292 on the held-out set and stops in 0 of 60 replies, so every reply runs to 512 tokens. TRL prints 1,934 packed rows and trains 121 steps. The logged training loss goes from 1.3071 to 1.0697 on seed 0. After SFT the held-out loss is 1.0870 and 52 of 60 replies stop. Training took 765 s in our run and 830 s in the notebook run, peaking at 7.03 GB.

In [ ]:
import sft_common as C            # written by the notebook's third cell; also labs/lltSFT/sft_common.py in the repo
GPU = torch.device("cuda")
BIG = "HuggingFaceTB/SmolLM2-360M"
tok360 = AutoTokenizer.from_pretrained(BIG)
tok360.chat_template, tok360.eos_token, tok360.pad_token = tok.chat_template, "<|im_end|>", "<|endoftext|>"

def fresh_model():
    return AutoModelForCausalLM.from_pretrained(BIG, dtype=torch.float32).to(GPU)   # fp32 master weights

def make_trainer(model, **changes):
    cfg = dict(output_dir="sft-360m", report_to="none", save_strategy="no",
               fp16=True, bf16=False,          # a T4 has no bf16 tensor cores, and SFTConfig defaults to bf16
               learning_rate=5e-5, lr_scheduler_type="cosine", warmup_steps=10, num_train_epochs=1,
               per_device_train_batch_size=2, gradient_accumulation_steps=8, max_length=1024,
               packing=True, assistant_only_loss=True, logging_steps=5, seed=0, data_seed=0)
    cfg.update(changes)
    data = Dataset.from_list([{"messages": c["messages"]} for c in train])
    return SFTTrainer(model=model, args=SFTConfig(**cfg), processing_class=tok360, train_dataset=data)

model360 = fresh_model()
gen_set = C.generation_set(held)                 # 20 held-out first turns from each split
before = C.eval_loss(model360, tok360, held, GPU, batch_size=1, dtype=torch.float16)["loss"]
g0 = C.eval_generate(model360, tok360, gen_set, GPU, dtype=torch.float16)
print("base | held-out loss", round(before, 4), "| stop rate", g0["stopRate"], "| mean new tokens", g0["meanNewTokens"])

In [ ]:
trainer = make_trainer(model360)
print(len(trainer.train_dataset), "packed rows of up to 1,024 tokens")
out = trainer.train()
logs = [h for h in trainer.state.log_history if "loss" in h]
print("steps", trainer.state.global_step, "| train loss", round(logs[0]["loss"], 4), "->", round(logs[-1]["loss"], 4),
      "| seconds", round(out.metrics["train_runtime"]), "| peak GB", round(torch.cuda.max_memory_allocated() / 1e9, 2))

model360.eval()
after = C.eval_loss(model360, tok360, held, GPU, batch_size=1, dtype=torch.float16)["loss"]
g1 = C.eval_generate(model360, tok360, gen_set, GPU, dtype=torch.float16)
print("SFT  | held-out loss", round(after, 4), "| stop rate", g1["stopRate"], "| mean new tokens", g1["meanNewTokens"])
print(g1["samples"][20]["text"][:400])

## Step 9. Train on every token

What you should see. Two every-token runs, held-out losses 1.1045 and 1.1035, against 1.0870 and 1.0871 with the mask. Their logged training loss starts higher, 1.98 and 2.15, and falls further. The stop rates overlap, 54 and 51 of 60 against 52 and 52.

In [ ]:
import gc, time
from transformers import TrainerCallback

RUN_EXPERIMENT = False      # True retrains on your T4; our runs are in the tables of steps 9 to 11

class EpochScore(TrainerCallback):
    def on_epoch_end(self, args, state, control, model=None, **kw):
        loss = C.eval_loss(model, tok360, held, GPU, batch_size=1, dtype=torch.float16)["loss"]
        print("epoch", round(state.epoch), "| step", state.global_step, "| held-out loss", round(loss, 4))

def train_and_score(trainer, name, each_epoch=False):
    if each_epoch:
        trainer.add_callback(EpochScore())
    t0 = time.time()
    trainer.train()
    sec = time.time() - t0
    trainer.model.eval()
    loss = C.eval_loss(trainer.model, tok360, held, GPU, batch_size=1, dtype=torch.float16)["loss"]
    g = C.eval_generate(trainer.model, tok360, gen_set, GPU, dtype=torch.float16)
    logs = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    print(name, "| steps", trainer.state.global_step, "| seconds", round(sec), "| train loss", round(logs[0], 4), "->",
          round(logs[-1], 4), "| held-out loss", round(loss, 4), "| stops", round(g["stopRate"] * 60), "of 60")
    return {"name": name, "steps": trainer.state.global_step, "seconds": round(sec, 1), "trainLoss": [logs[0], logs[-1]],
            "heldOut": loss, "stopRate": g["stopRate"], "meanNewTokens": g["meanNewTokens"]}

def fresh_trainer(**changes):
    gc.collect()
    torch.cuda.empty_cache()                       # the previous run's weights and optimizer state go first
    return make_trainer(fresh_model(), **changes)

if RUN_EXPERIMENT:
    del trainer, model360                          # free step 8's model before the next one loads
    train_and_score(fresh_trainer(assistant_only_loss=False), "every token, seed 0")

## Step 10. Pack naively, or pad

What you should see. The naive runs end at 1.0920 and 1.0925 against 1.0870 and 1.0871, and each step takes 14.1 to 14.6 s against 6.1 to 6.3 s. The padded run ends at 1.0893 after 913 s, against 736 to 765 s for the masked packed runs. Stop rates stay between 52 and 54 of 60.

In [ ]:
def cross_doc_collator(collate, restart_positions, explicit_mask=True):
    # Same packed rows and labels as TRL's collator, but every token may attend to every earlier token in the row.
    # explicit_mask=True passes a full causal mask, so SDPA gets an explicit mask, as it does with the document mask.
    def wrapped(examples):
        b = collate(examples)
        n = b["input_ids"].shape[1]
        if not restart_positions:
            b["position_ids"] = torch.arange(n)[None]
        if explicit_mask:
            b["attention_mask"] = torch.tril(torch.ones(n, n, dtype=torch.bool))[None, None]
        for k in ("cu_seq_lens_q", "cu_seq_lens_k", "max_length_q", "max_length_k", "seq_idx"):
            b.pop(k, None)
        return b
    return wrapped

if RUN_EXPERIMENT:
    for name, restart, explicit in [("naive packing", False, False),
                                    ("cross-document, positions restart", True, True),
                                    ("cross-document, positions run on", False, True)]:
        t = fresh_trainer()
        t.data_collator = cross_doc_collator(t.data_collator, restart, explicit)
        train_and_score(t, name)
        del t
    train_and_score(fresh_trainer(packing=False, per_device_train_batch_size=4), "padded")

## Step 11. Train for three epochs

What you should see. Held-out loss 1.0599, 1.0455 and 1.0532 after epochs 1, 2 and 3. The logged training loss ends at 0.790. The run took 2,186 s of training for 363 steps.

In [ ]:
if RUN_EXPERIMENT:
    train_and_score(fresh_trainer(num_train_epochs=3), "3 epochs", each_epoch=True)

## Prove it 1

Write `build_labels`, `masked_mean_nll` and `doc_mask` in NumPy for a toy chat tokenizer with the SmolLM2 turn layout. Run the cell, and it prints All checks passed when all three are right. The checks include a two-reply chat, a chat with no assistant turn and a single document.

In [ ]:
import numpy as np

# A toy chat tokenizer. Special ids first, then one id per word.
IM_START, IM_END, NL = 1, 2, 3
ROLE = {"system": 4, "user": 5, "assistant": 6}
WORDS = {}

def word_id(w):
    return WORDS.setdefault(w, 10 + len(WORDS))

def render(chat):
    # Each turn: <|im_start|> role NL words... <|im_end|> NL, as in the SmolLM2 template
    ids = []
    for role, text in chat:
        ids += [IM_START, ROLE[role], NL] + [word_id(w) for w in text.split()] + [IM_END, NL]
    return np.array(ids)

def build_labels(ids):
    # Return an array like ids: the token id inside each assistant reply and its <|im_end|>, -100 everywhere else.
    # The header IM_START, ROLE["assistant"], NL and the NL after <|im_end|> get -100.
    raise NotImplementedError

def masked_mean_nll(logp, labels):
    # logp: (T, V) log-probabilities; row t predicts token t + 1. Mean NLL over the labels that are not -100.
    raise NotImplementedError

def doc_mask(lengths):
    # (T, T) boolean: True where query i may attend to key j. Causal, and only inside its own document.
    raise NotImplementedError

# Checks. Run the cell; it prints All checks passed when all three functions are right.
chat = [("system", "be brief"), ("user", "name a fruit"), ("assistant", "a mango"),
        ("user", "and a colour"), ("assistant", "red")]
ids = render(chat)
labels = build_labels(ids)
assert labels.shape == ids.shape, "return one label per token"
assert labels.tolist() == [-100] * 18 + [WORDS["a"], WORDS["mango"], IM_END] + [-100] * 12 + [WORDS["red"], IM_END, -100], labels.tolist()
assert (labels != -100).sum() == 5, "two replies of 2 and 1 words, each with its <|im_end|>"
assert build_labels(render([("user", "hi"), ("user", "again")])).max() == -100, "no assistant turn means no loss"
V = 40
logits = np.sin(np.arange(len(ids) * V).reshape(len(ids), V) * 0.37)
logp = logits - np.log(np.exp(logits).sum(axis=1, keepdims=True))
loss = masked_mean_nll(logp, labels)
assert abs(loss - 4.12037) < 1e-4, round(loss, 5)
assert abs(masked_mean_nll(logp, ids) - 3.86735) < 1e-4, "with labels = ids every next token counts: " + str(round(masked_mean_nll(logp, ids), 5))
m = doc_mask([3, 2])
assert m.dtype == bool and m.shape == (5, 5), "a (T, T) boolean array"
assert m.astype(int).tolist() == [[1, 0, 0, 0, 0], [1, 1, 0, 0, 0], [1, 1, 1, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 1, 1]], m.astype(int).tolist()
assert doc_mask([4]).sum() == 10, "one document is the plain causal mask"
print("loss tokens", int((labels != -100).sum()), "of", len(ids), "| masked NLL", round(loss, 4))
print("All checks passed")